# Varying Intercepts: The Core Hierarchical Model

## Learning Objectives

By the end of this notebook you will be able to:

1. Write down the **varying-intercept model** in full mathematical notation and explain each level of the hierarchy.
2. Explain the role of **hyperpriors** ($\bar{\alpha}$, $\tau$, $\sigma$) and justify standard prior choices (half-Cauchy, exponential).
3. Derive both the **centered** and **non-centered** parameterisations, and explain *why* non-centered sampling is more efficient.
4. Describe the **funnel geometry** problem and recognise it in diagnostics.
5. Fit a varying-intercept model in **PyMC** with both parameterisations and compare their sampling efficiency.
6. Interpret posterior summaries: group-level estimates, population distribution, and convergence diagnostics.

### Prerequisites

- Notebook 01 of this module (partial pooling, shrinkage, exchangeability).
- Module 07 (MCMC, NUTS, convergence diagnostics).

In [ ]:
import sys, os, shutil
from pathlib import Path

os.environ["PYTENSOR_FLAGS"] = "device=cpu,floatX=float64,cxx="

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        "sudo apt-get update -qq && sudo apt-get install -y -qq "
        "libcairo2-dev libpango1.0-dev && pip install -q manim pymc arviz ipython==8.21.0"
    )

_miktex_bin = Path.home() / "AppData/Local/Programs/MiKTeX/miktex/bin/x64"
if _miktex_bin.exists() and str(_miktex_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + str(_miktex_bin)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, os.path.abspath("../../src"))
from amstats.plotting import apply_style

apply_style()

HAS_PYMC = False
try:
    import pymc as pm
    import arviz as az
    HAS_PYMC = True
    print(f"PyMC {pm.__version__}, ArviZ {az.__version__}")
except Exception as e:
    print(f"PyMC not available: {type(e).__name__}: {e}")


class Cfg:
    root = Path("../../").resolve()
    gif_dir = root / "media" / "gifs"
    has_latex: bool = (
        shutil.which("latex") is not None or shutil.which("pdflatex") is not None
    )

    def __init__(self):
        self.gif_dir.mkdir(parents=True, exist_ok=True)

    def apply_manim_config(self):
        from manim import config as mcfg
        mcfg.format = "gif"

    def math_text(self, expr, **kwargs):
        from manim import MathTex, Text
        if self.has_latex:
            return MathTex(expr, **kwargs)
        return Text(expr, **kwargs)

    def save_gifs(self, clean=True):
        local_media = Path("media")
        found = list(local_media.rglob("*.gif")) if local_media.exists() else []
        if not found:
            print("  No new GIFs to save.")
            return
        for gif in found:
            dest = self.gif_dir / gif.name
            shutil.copy2(gif, dest)
            print(f"  \u2713 media/gifs/{gif.name}")
        if clean:
            for sub in ("videos", "images", "Tex"):
                d = local_media / sub
                if d.exists():
                    shutil.rmtree(d, ignore_errors=True)
            print("  Cleaned up local temp render files (kept media/jupyter/).")


cfg = Cfg()
rng = np.random.default_rng(42)

---

## 1. The Varying-Intercept Model

In the previous notebook we motivated hierarchical models using the simplest case — estimating group means.  Now we formalise the **varying-intercept regression model**, which is the workhorse of multilevel analysis.

### Setup

We have $N$ observations indexed by $i = 1, \ldots, N$, and each observation belongs to one of $J$ groups.  Let $j[i]$ denote the group membership of observation $i$.  We also have $p$ predictor variables $\mathbf{x}_i = (x_{i1}, \ldots, x_{ip})^\top$.

### The Model

**Level 1 — Within groups (likelihood):**

$$y_i \mid \alpha_{j[i]}, \boldsymbol{\beta}, \sigma \;\sim\; \mathcal{N}\!\left(\alpha_{j[i]} + \mathbf{x}_i^\top \boldsymbol{\beta}, \;\; \sigma^2\right)$$

Each group $j$ has its own intercept $\alpha_j$, but all groups share the same slope coefficients $\boldsymbol{\beta}$ and noise variance $\sigma^2$.

**Level 2 — Between groups:**

$$\alpha_j \mid \bar{\alpha}, \tau \;\sim\; \mathcal{N}(\bar{\alpha}, \;\tau^2), \qquad j = 1, \ldots, J$$

The group intercepts are drawn from a shared **population distribution** with mean $\bar{\alpha}$ and standard deviation $\tau$.

**Hyperpriors:**

$$\bar{\alpha} \sim \mathcal{N}(0, 5^2), \qquad \tau \sim \text{Half-Cauchy}(0, 2), \qquad \sigma \sim \text{Exponential}(1), \qquad \beta_k \sim \mathcal{N}(0, 2^2)$$

### Interpretation

- $\bar{\alpha}$: the **average intercept** across all groups.
- $\tau$: the **between-group standard deviation** — how much the intercepts vary from group to group.
- $\sigma$: the **within-group (residual) standard deviation** — how much individual observations vary within a group.
- $\alpha_j$: the intercept for group $j$, a compromise between the group's own data and the population mean $\bar{\alpha}$.

The key modelling choice is that the **intercepts vary by group** but the **slopes are shared**. This is appropriate when you expect groups to differ in their baseline level but respond to predictors in the same way. We relax this assumption in the next notebook.

### Why Not Just Use Dummy Variables?

A natural alternative is to include a **fixed effect** for each group — that is, add $J - 1$ dummy variables to a standard regression:

$$y_i = \alpha_1 \cdot \mathbb{1}[j[i]=1] + \cdots + \alpha_J \cdot \mathbb{1}[j[i]=J] + \mathbf{x}_i^\top \boldsymbol{\beta} + \varepsilon_i$$

This is the **no-pooling** (fixed-effects) approach.  It works, but has three problems:

1. **Overfitting for small groups.** A group with $n_j = 3$ observations gets its own unconstrained intercept — pure noise.
2. **No information sharing.** What we learn about School 1 tells us nothing about School 2, even though they are similar institutions.
3. **Cannot predict for new groups.** If a new school appears, the fixed-effects model has no estimate for it. The hierarchical model can use the population distribution $\mathcal{N}(\bar{\alpha}, \tau^2)$ as a prior.

The hierarchical model solves all three problems simultaneously.

---

## 2. Prior Choices for Scale Parameters

The choice of prior on $\tau$ (the between-group SD) deserves careful discussion, because $\tau$ controls the degree of pooling.

### The Half-Cauchy Prior

The **half-Cauchy** distribution, $\tau \sim \text{Half-Cauchy}(0, \beta)$, has density:

$$p(\tau) = \frac{2}{\pi \beta \left(1 + (\tau/\beta)^2\right)}, \qquad \tau > 0$$

It was recommended by Gelman (2006) as a default prior for group-level standard deviations because:

- It is **proper** (integrable), unlike the improper $p(\tau) \propto 1/\tau$.
- It has a **heavy tail**, so it doesn't prevent $\tau$ from being large if the data require it.
- It has a **mode at zero**, providing gentle regularisation toward pooling.
- It has only one hyperparameter $\beta$ (the scale), which controls how "wide" the prior is.

### The Exponential Prior

An alternative is $\tau \sim \text{Exponential}(\lambda)$, which is somewhat more informative (thinner tail) but still a reasonable default when you expect moderate between-group variation.

### Why Not Inverse-Gamma?

Older textbooks use inverse-gamma priors on $\tau^2$ for conjugacy.  This is **not recommended** because the inverse-gamma with small shape parameters places too much mass near zero, causing the posterior of $\tau$ to pile up at zero artificially.  The half-Cauchy avoids this problem.

---

## 3. Centered vs. Non-Centered Parameterisation

This section covers the most important computational concept in hierarchical modelling.  The way we *write* the model affects how well MCMC samplers explore the posterior — even though both parameterisations define the same model.

### Centered Parameterisation (CP)

The model as written above is the centered parameterisation:

$$\alpha_j \sim \mathcal{N}(\bar{\alpha}, \tau^2)$$

Here $\alpha_j$ is sampled directly from the population distribution.  The sampler must simultaneously explore:
- The value of each $\alpha_j$
- The value of $\tau$

### Non-Centered Parameterisation (NCP)

We introduce auxiliary variables $z_j \sim \mathcal{N}(0, 1)$ and recover $\alpha_j$ via a deterministic transformation:

$$z_j \sim \mathcal{N}(0, 1), \qquad \alpha_j = \bar{\alpha} + z_j \cdot \tau$$

Mathematically, this is identical: if $z_j \sim \mathcal{N}(0,1)$, then $\bar{\alpha} + z_j \cdot \tau \sim \mathcal{N}(\bar{\alpha}, \tau^2)$.

But the **geometry** of the posterior is very different.

### The Funnel Problem

Consider the joint posterior $p(\tau, \alpha_j \mid \text{data})$ in the centered parameterisation.  When $\tau$ is small, all $\alpha_j$ must be close to $\bar{\alpha}$ — the posterior forms a **narrow funnel** near $\tau = 0$.  When $\tau$ is large, the $\alpha_j$ can spread out — the posterior is wide.

This funnel shape is pathological for MCMC:
- The sampler needs **small step sizes** to navigate the narrow neck of the funnel (near $\tau \approx 0$).
- But those same small steps mean **slow exploration** of the wide body (large $\tau$).
- The result: low effective sample size, high autocorrelation, potential divergences.

In the **non-centered** parameterisation, we sample $(\tau, z_j)$ instead of $(\tau, \alpha_j)$.  Since $z_j \sim \mathcal{N}(0,1)$ does not depend on $\tau$, the posterior in $(\tau, z_j)$-space has no funnel — it is much more regular, and NUTS can explore it efficiently.

### When to Use Which?

| Scenario | Recommended |
|----------|-------------|
| Few data per group, many groups | **Non-centered** |
| Lots of data per group | **Centered** (data overwhelms the prior, so the funnel is weak) |
| Not sure | **Non-centered** (safer default) |

The following animation shows the funnel problem. In the centered parameterisation, the joint posterior of $(\tau, \alpha_j)$ forms a funnel: the width of the distribution of $\alpha_j$ shrinks as $\tau \to 0$. The non-centered parameterisation replaces $\alpha_j$ with $z_j$, which is independent of $\tau$, producing a much more sampler-friendly geometry.

In [ ]:
from manim import *

cfg.apply_manim_config()

In [ ]:
%%manim -qm -v WARNING FunnelGeometry

class FunnelGeometry(Scene):
    """Visualise the funnel in centered vs non-centered parameterisation."""

    def construct(self):
        # Title
        title = Text("The Funnel Problem", font_size=30).to_edge(UP, buff=0.3)
        self.play(Write(title))

        # ── Left panel: Centered parameterisation ──
        left_label = Text("Centered", font_size=22, color=RED).move_to(LEFT * 3.5 + UP * 2.5)
        ax_left = Axes(
            x_range=[-4, 4, 1], y_range=[0, 3, 1],
            x_length=5, y_length=3,
            axis_config={"stroke_width": 1.5, "include_ticks": False},
        ).move_to(LEFT * 3.2 + DOWN * 0.3)
        x_lab = cfg.math_text(r"\alpha_j", font_size=22).next_to(ax_left.x_axis, DOWN, buff=0.15)
        y_lab = cfg.math_text(r"\tau", font_size=22).next_to(ax_left.y_axis, LEFT, buff=0.15)

        self.play(Write(left_label), Create(ax_left), Write(x_lab), Write(y_lab))

        # Generate funnel samples
        np.random.seed(0)
        n_pts = 600
        tau_samp = np.abs(np.random.standard_cauchy(n_pts))
        tau_samp = tau_samp[tau_samp < 3.0][:400]
        alpha_samp = np.random.normal(0, tau_samp)

        funnel_dots = VGroup(*[
            Dot(
                ax_left.c2p(a, t),
                radius=0.02, color=RED, fill_opacity=0.5
            )
            for a, t in zip(alpha_samp, tau_samp)
        ])

        self.play(FadeIn(funnel_dots, lag_ratio=0.005), run_time=2)

        # Annotation: narrow neck
        arrow_neck = Arrow(
            ax_left.c2p(2.5, 0.3), ax_left.c2p(0.5, 0.15),
            color=YELLOW, stroke_width=2, max_tip_length_to_length_ratio=0.15
        )
        neck_text = Text("Narrow neck\n(hard to sample)", font_size=14, color=YELLOW
        ).next_to(arrow_neck.get_start(), RIGHT, buff=0.1)
        self.play(GrowArrow(arrow_neck), Write(neck_text))
        self.wait(1)

        # ── Right panel: Non-centered parameterisation ──
        right_label = Text("Non-Centered", font_size=22, color=GREEN).move_to(RIGHT * 3.5 + UP * 2.5)
        ax_right = Axes(
            x_range=[-4, 4, 1], y_range=[0, 3, 1],
            x_length=5, y_length=3,
            axis_config={"stroke_width": 1.5, "include_ticks": False},
        ).move_to(RIGHT * 3.2 + DOWN * 0.3)
        z_lab = cfg.math_text(r"z_j", font_size=22).next_to(ax_right.x_axis, DOWN, buff=0.15)
        y_lab2 = cfg.math_text(r"\tau", font_size=22).next_to(ax_right.y_axis, LEFT, buff=0.15)

        self.play(Write(right_label), Create(ax_right), Write(z_lab), Write(y_lab2))

        # Non-centered: z ~ N(0,1) independent of tau
        z_samp = np.random.normal(0, 1, len(tau_samp))

        ncp_dots = VGroup(*[
            Dot(
                ax_right.c2p(z, t),
                radius=0.02, color=GREEN, fill_opacity=0.5
            )
            for z, t in zip(z_samp, tau_samp)
        ])

        self.play(FadeIn(ncp_dots, lag_ratio=0.005), run_time=2)

        ok_text = Text("No funnel!\n(easy to sample)", font_size=14, color=GREEN
        ).move_to(RIGHT * 5.5 + DOWN * 0.3)
        self.play(Write(ok_text))

        # Bottom: the transformation
        transform_eq = cfg.math_text(
            r"\alpha_j = \bar{\alpha} + z_j \cdot \tau",
            font_size=24
        ).to_edge(DOWN, buff=0.4)
        box = SurroundingRectangle(transform_eq, color=YELLOW, buff=0.15)
        self.play(Write(transform_eq), Create(box))
        self.wait(2)

In [ ]:
cfg.save_gifs(clean=True)

---

## 4. Plate Notation

Hierarchical models are often represented using **plate diagrams** (also called Bayesian network diagrams). A plate is a rectangle that denotes replication — the nodes inside are repeated for each index.

For the varying-intercept model:

```
      ┌──────────────────────────────────────┐
      │  μ_α         τ          σ            │  Hyperpriors
      │   │           │          │            │
      │   ▼           ▼          │            │
      │  ᾱ ────────► α_j ◄──── τ            │
      │               │                       │
      │  ┌───────────────────────────┐        │
      │  │            │              │ N      │
      │  │  x_i ───► y_i ◄── σ     │        │
      │  │                           │        │
      │  └───────────────────────────┘        │
      │         (observations)                │
      │  ┌───────────────────────┐            │
      │  │      α_j              │ J          │
      │  └───────────────────────┘            │
      └──────────────────────────────────────┘
```

- **Outer plate** ($J$): one $\alpha_j$ per group.
- **Inner plate** ($N$): one observation $y_i$ per data point.
- **Shaded nodes** ($y_i$, $x_i$): observed data.
- **Open nodes** ($\bar{\alpha}$, $\tau$, $\sigma$, $\alpha_j$): parameters to be estimated.
- **Arrows**: conditional dependencies.

---

## 5. Worked Example: Cheese Ratings

We now fit the varying-intercept model to a concrete dataset.  We simulate data inspired by the cheese-tasting experiment from the Bayesian Statistics reference material: tasters from different **backgrounds** (urban, rural) rate different **cheese types** on a 1–10 scale.  The groups are the background categories, and we want to estimate how the baseline rating differs by background.

For pedagogical clarity we simulate the data from a known model, so we can verify that the posterior recovers the true parameters.

### Data-Generating Process

$$\text{rating}_{ij} = \alpha_{\text{bg}[i]} + \beta \cdot \text{cheese\_quality}_i + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)$$

$$\alpha_j \sim \mathcal{N}(\bar{\alpha}, \tau^2), \qquad j = 1, \ldots, 4 \text{ (backgrounds)}$$

In [ ]:
# ── Simulate cheese-tasting data ──
J = 4  # backgrounds: urban, suburban, rural, metropolitan
bg_names = ["Urban", "Suburban", "Rural", "Metropolitan"]

# True parameters
alpha_bar_true = 5.5     # grand mean rating
tau_true = 1.2           # between-background SD
sigma_true = 1.0         # within-group noise
beta_true = 0.8          # effect of cheese quality

# Group sizes (unbalanced to show differential shrinkage)
n_per_group = [8, 15, 50, 12]
N = sum(n_per_group)

# True group intercepts
alpha_true = rng.normal(alpha_bar_true, tau_true, J)
print("True group intercepts:", {bg: f"{a:.2f}" for bg, a in zip(bg_names, alpha_true)})

# Generate data
group_idx = np.concatenate([np.full(n, j, dtype=int) for j, n in enumerate(n_per_group)])
cheese_quality = rng.normal(0, 1, N)  # standardised predictor
mu = alpha_true[group_idx] + beta_true * cheese_quality
rating = mu + rng.normal(0, sigma_true, N)

print(f"\nTotal observations: {N}")
print(f"Group sizes: {dict(zip(bg_names, n_per_group))}")

### 5.1 Centered Parameterisation in PyMC

In [ ]:
if HAS_PYMC:
    with pm.Model() as centered_model:
        # Hyperpriors
        a_bar = pm.Normal("a_bar", mu=0, sigma=5)
        tau = pm.HalfCauchy("tau", beta=2)
        sigma = pm.Exponential("sigma", lam=1)

        # Slope (shared across groups)
        beta = pm.Normal("beta", mu=0, sigma=2)

        # Group intercepts — CENTERED
        alpha = pm.Normal("alpha", mu=a_bar, sigma=tau, shape=J)

        # Likelihood
        mu_obs = alpha[group_idx] + beta * cheese_quality
        y_obs = pm.Normal("y_obs", mu=mu_obs, sigma=sigma, observed=rating)

        # Sample
        trace_cp = pm.sample(2000, tune=1000, chains=4, random_seed=42,
                             target_accept=0.9, return_inferencedata=True)

    print("\n── Centered parameterisation ──")
    print(az.summary(trace_cp, var_names=["a_bar", "tau", "sigma", "beta", "alpha"]))
else:
    print("PyMC not available.")

### 5.2 Non-Centered Parameterisation in PyMC

In [ ]:
if HAS_PYMC:
    with pm.Model() as noncentered_model:
        # Hyperpriors
        a_bar = pm.Normal("a_bar", mu=0, sigma=5)
        tau = pm.HalfCauchy("tau", beta=2)
        sigma = pm.Exponential("sigma", lam=1)

        # Slope (shared)
        beta = pm.Normal("beta", mu=0, sigma=2)

        # Group intercepts — NON-CENTERED
        z = pm.Normal("z", mu=0, sigma=1, shape=J)
        alpha = pm.Deterministic("alpha", a_bar + z * tau)

        # Likelihood
        mu_obs = alpha[group_idx] + beta * cheese_quality
        y_obs = pm.Normal("y_obs", mu=mu_obs, sigma=sigma, observed=rating)

        # Sample
        trace_ncp = pm.sample(2000, tune=1000, chains=4, random_seed=42,
                              target_accept=0.9, return_inferencedata=True)

    print("\n── Non-centered parameterisation ──")
    print(az.summary(trace_ncp, var_names=["a_bar", "tau", "sigma", "beta", "alpha"]))
else:
    print("PyMC not available.")

### 5.3 Comparing Sampling Efficiency

The key diagnostic is the **effective sample size (ESS)** — how many effectively independent samples the chain produced.  Higher is better.  We also check for **divergences** — transitions where the sampler's numerical integration failed, indicating geometric pathology.

In [ ]:
if HAS_PYMC:
    # Extract ESS for alpha parameters
    summary_cp = az.summary(trace_cp, var_names=["alpha"])
    summary_ncp = az.summary(trace_ncp, var_names=["alpha"])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: ESS comparison
    ax = axes[0]
    x = np.arange(J)
    w = 0.35
    ax.bar(x - w/2, summary_cp["ess_bulk"].values, w, color="red", alpha=0.7, label="Centered")
    ax.bar(x + w/2, summary_ncp["ess_bulk"].values, w, color="green", alpha=0.7, label="Non-centered")
    ax.set_xticks(x)
    ax.set_xticklabels(bg_names, fontsize=9)
    ax.set_ylabel("Bulk ESS")
    ax.set_title("Effective Sample Size: Centered vs Non-Centered")
    ax.legend()

    # Right: Posterior of alpha overlaid with truth
    ax = axes[1]
    for j_idx in range(J):
        samples = trace_ncp.posterior["alpha"].values[:, :, j_idx].flatten()
        ax.hist(samples, bins=40, density=True, alpha=0.4, label=bg_names[j_idx])
        ax.axvline(alpha_true[j_idx], color="black", linestyle="--", linewidth=1)

    ax.set_xlabel(r"$\alpha_j$ (group intercept)")
    ax.set_ylabel("Density")
    ax.set_title("Posterior of Group Intercepts (NCP)\nDashed = true values")
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    # Divergences
    div_cp = trace_cp.sample_stats["diverging"].values.sum()
    div_ncp = trace_ncp.sample_stats["diverging"].values.sum()
    print(f"\nDivergences — Centered: {div_cp}, Non-centered: {div_ncp}")

In this example (with only 4 groups and moderate data), both parameterisations should work reasonably well. The non-centered version typically shows equal or higher ESS. The difference becomes dramatic when $J$ is large and $n_j$ is small — the funnel becomes more pronounced and the centered parameterisation may produce many divergences.

---

## 6. Interpreting the Results

### 6.1 Group-Level Estimates and Shrinkage

In [ ]:
if HAS_PYMC:
    # Posterior means for group intercepts
    alpha_post = trace_ncp.posterior["alpha"].values.reshape(-1, J)
    alpha_post_mean = alpha_post.mean(axis=0)
    alpha_post_lo = np.percentile(alpha_post, 5.5, axis=0)
    alpha_post_hi = np.percentile(alpha_post, 94.5, axis=0)

    # No-pooling estimates (group sample means after removing slope effect)
    beta_post_mean = trace_ncp.posterior["beta"].values.flatten().mean()
    residuals = rating - beta_post_mean * cheese_quality
    alpha_nopool = np.array([residuals[group_idx == j].mean() for j in range(J)])

    # Population mean
    abar_post_mean = trace_ncp.posterior["a_bar"].values.flatten().mean()

    fig, ax = plt.subplots(figsize=(9, 5))

    for j_idx in range(J):
        # Arrow from no-pool to partial-pool
        ax.annotate(
            "", xy=(alpha_post_mean[j_idx], j_idx), xytext=(alpha_nopool[j_idx], j_idx),
            arrowprops=dict(arrowstyle="->", color="steelblue", lw=2)
        )
        ax.scatter(alpha_nopool[j_idx], j_idx, color="red", s=80, zorder=5)
        ax.scatter(alpha_post_mean[j_idx], j_idx, color="green", s=80, zorder=5)
        ax.plot([alpha_post_lo[j_idx], alpha_post_hi[j_idx]], [j_idx, j_idx],
                color="green", linewidth=2, alpha=0.5)
        ax.scatter(alpha_true[j_idx], j_idx, color="black", s=60, marker="x", zorder=6)
        ax.text(max(alpha_nopool[j_idx], alpha_post_mean[j_idx]) + 0.25, j_idx,
                f"{bg_names[j_idx]} (n={n_per_group[j_idx]})", fontsize=10, va="center")

    ax.axvline(abar_post_mean, color="gold", linestyle="--", linewidth=2, label=r"$\bar{\alpha}$ (posterior mean)")
    ax.scatter([], [], color="red", label="No-pooling estimate")
    ax.scatter([], [], color="green", label="Partial-pooling (posterior mean)")
    ax.scatter([], [], color="black", marker="x", label="True value")
    ax.set_xlabel(r"$\alpha_j$")
    ax.set_yticks([])
    ax.set_title("Shrinkage: No-Pooling → Partial-Pooling Estimates")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.show()

The pattern is exactly as predicted by the theory:
- The **Urban** group ($n = 8$) shrinks substantially toward the grand mean.
- The **Rural** group ($n = 50$) barely moves — it has enough data to stand on its own.
- The partial-pooling estimates (green) are typically **closer to the true values** (black crosses) than the no-pooling estimates (red), especially for small groups.

### 6.2 Convergence Diagnostics

In [ ]:
if HAS_PYMC:
    az.plot_trace(trace_ncp, var_names=["a_bar", "tau", "sigma", "beta"],
                  compact=True, figsize=(12, 6))
    plt.suptitle("Trace Plots (Non-Centered Model)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

Good traces show:
- **Left panels (posterior):** Smooth, unimodal distributions for each parameter.
- **Right panels (trace):** "hairy caterpillar" appearance — chains mix well with no visible trends or stuck regions.
- **All four chains overlap** — they have converged to the same target distribution.

---

## 7. The Population Distribution: Prediction for New Groups

One of the most powerful features of hierarchical models is the ability to make predictions for **groups not in the data**.

If a new background category appears (say, "Semi-rural"), we can predict its intercept using the **population distribution**:

$$\alpha_{\text{new}} \sim \mathcal{N}(\bar{\alpha}, \tau^2)$$

In practice, we draw from the posterior predictive:

$$p(\alpha_{\text{new}} \mid \text{data}) = \int \mathcal{N}(\alpha_{\text{new}} \mid \bar{\alpha}, \tau^2) \, p(\bar{\alpha}, \tau \mid \text{data}) \, d\bar{\alpha} \, d\tau$$

which we approximate by sampling: for each posterior draw of $(\bar{\alpha}, \tau)$, generate $\alpha_{\text{new}} \sim \mathcal{N}(\bar{\alpha}, \tau^2)$.

In [ ]:
if HAS_PYMC:
    # Posterior predictive for a new group
    abar_samples = trace_ncp.posterior["a_bar"].values.flatten()
    tau_samples = trace_ncp.posterior["tau"].values.flatten()
    alpha_new = rng.normal(abar_samples, tau_samples)

    fig, ax = plt.subplots(figsize=(9, 4))

    # Existing group posteriors
    for j_idx in range(J):
        samples = trace_ncp.posterior["alpha"].values[:, :, j_idx].flatten()
        ax.hist(samples, bins=40, density=True, alpha=0.3, label=bg_names[j_idx])

    # New group prediction
    ax.hist(alpha_new, bins=50, density=True, alpha=0.6, color="black",
            histtype="step", linewidth=2, label="New group (prediction)")

    ax.set_xlabel(r"$\alpha_j$")
    ax.set_ylabel("Density")
    ax.set_title("Posterior for Existing Groups + Prediction for a New Group")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"\nNew group prediction: mean = {alpha_new.mean():.2f}, "
          f"89% CI = [{np.percentile(alpha_new, 5.5):.2f}, {np.percentile(alpha_new, 94.5):.2f}]")

The new-group prediction (black outline) is wider than any existing group's posterior — it reflects our uncertainty about where a new group would fall.  This is much more honest than the no-pooling approach, which has *nothing* to say about new groups.

---

## 8. Key Takeaways

1. The **varying-intercept model** is the simplest and most widely used hierarchical model: groups share slopes but have their own intercepts drawn from a population distribution.

2. The **non-centered parameterisation** ($\alpha_j = \bar{\alpha} + z_j \cdot \tau$, with $z_j \sim \mathcal{N}(0,1)$) avoids the funnel geometry that plagues the centered version.  Use non-centered as the default.

3. **Priors on scale parameters** matter: half-Cauchy or exponential are recommended over inverse-gamma.

4. Hierarchical models can **predict for new groups** using the population distribution — something no-pooling models cannot do.

5. Always check **convergence diagnostics**: trace plots, ESS, R-hat, and divergences.

## Further Reading

- Gelman, A. (2006). *Prior distributions for variance parameters in hierarchical models*. Bayesian Analysis.
- Betancourt, M. & Girolami, M. (2015). *Hamiltonian Monte Carlo for hierarchical models*. Current Trends in Bayesian Methodology.
- McElreath, R. (2020). *Statistical Rethinking*, Ch. 13.

**Next notebook:** We extend the model to allow **slopes** to vary across groups, introducing correlated effects, the LKJ prior, and Cholesky decomposition.